# SBA Structure Identification

This notebook loads the prepared SBA dataset, uses the duplicated partition columns (`dataset_partition` and `dataset_split`) to define train/validation/test subsets, and runs the `structure_id` package on the training partition to discover explicit and latent structure.


In [1]:
from pathlib import Path
import pandas as pd
import sys

current_dir = Path.cwd().resolve()
candidate_roots = [current_dir, current_dir.parent] if current_dir.name == 'notebooks' else [current_dir]
project_root = next((root for root in candidate_roots if (root / 'data' / 'model_ready_numeric_data.csv').exists()), current_dir)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

data_path = project_root / 'data' / 'model_ready_numeric_data.csv'
if not data_path.exists():
    raise FileNotFoundError(f'Data file was not found at {data_path}')

df = pd.read_csv(data_path)
print('Columns:', df.columns[:8].tolist(), '...')
print('Partition columns:', df.columns[:2].tolist())
print(df[['dataset_partition', 'dataset_split']].head())
print('\nPartition counts:')
print(df['dataset_partition'].value_counts(dropna=False).to_string())


Columns: ['dataset_partition', 'dataset_split', 'loan_status_r', 'projectcounty_rcs_te', 'naicscode_rcs_te', 'franchisename_rcs_te', 'grossapproval', 'jobssupported'] ...
Partition columns: ['dataset_partition', 'dataset_split']
  dataset_partition dataset_split
0             train         train
1        validation    validation
2              test          test
3            active        active
4        validation    validation

Partition counts:
dataset_partition
active        55594
train         36297
test          12100
validation    12099


In [2]:
# Use the partition columns to select the data split. The first two columns duplicate the same information,
# so either can be used to define training/validation/test membership.
partition_col = 'dataset_partition'
split_col = 'dataset_split'

assert (df[partition_col] == df[split_col]).all(), 'The partition columns should match.'

train_df = df[df[partition_col] == 'train'].copy()
validation_df = df[df[partition_col] == 'validation'].copy()
test_df = df[df[partition_col] == 'test'].copy()

print(f'train rows: {len(train_df)}')
print(f'validation rows: {len(validation_df)}')
print(f'test rows: {len(test_df)}')

train_df = train_df.drop(columns=[partition_col, split_col])
validation_df = validation_df.drop(columns=[partition_col, split_col])
test_df = test_df.drop(columns=[partition_col, split_col])

print(train_df.head(2).to_string(index=False))


train rows: 36297
validation rows: 12099
test rows: 12100
 loan_status_r  projectcounty_rcs_te  naicscode_rcs_te  franchisename_rcs_te  grossapproval  jobssupported  franchisecode_rcs_te  borrower_longitude  sbadistrictoffice_rcs_te  congressionaldistrict_rcs_te  businessage_rcs_te  borrower_latitude  naicscode  congressionaldistrict  processingmethod_rcs_te
           0.0              0.046809          0.037548              0.080000         810000              0              0.080000            -89.6499                  0.032389                      0.022388            0.023259            39.1793   721110.0                   13.0                 0.034466
           0.0              0.029851          0.038221              0.024921         757000             40              0.024923           -105.0918                  0.022727                      0.025454            0.023259            39.7123   332510.0                    1.0                 0.011278


In [3]:
from structure_id import structure_id

result = structure_id(
    data=train_df,
    target='loan_status_r',
    task_type='classification',
    structural_covariates=[],
)

print(result.summary())
print('\nSelected columns:')
print(result.selected_columns)
print('\nExplicit column structures:')
for block in result.explicit_column_structures:
    print(block.describe())
    if block.merge_history:
        print('  Merge details:')
        for item in block.merge_history:
            left, right = item["merged_pair"]
            rho = float(item["spearman_rho"])
            print(f"    {left} <-> {right}: spearman rho = {rho:.6f}, distance={1.0 - abs(rho):.6f}")
        print('\n  Correlation matrix:')
        print(train_df[list(block.cluster_members)].corr(method="spearman"))
    else:
        print('  No merge history available.')
print('\nLatent structure:')
if result.latent_break is None:
    print('No statistically significant latent structure detected.')
else:
    print(result.latent_break)


Structure Identification Summary
Target: loan_status_r
Task type: classification
Provisional candidates: 14
Selected candidates: 14
Explicit column blocks: 1
Explicit row clusters: 4
Explicit column block details:
  - Cluster 1: members=['franchisename_rcs_te', 'franchisecode_rcs_te'], representative=franchisename_rcs_te, dropped=['franchisecode_rcs_te'], merge_pairs=[franchisename_rcs_te↔franchisecode_rcs_te (rho=0.9967)]
Instance structure summary:
  noise=57
  cluster_0=35962
  cluster_1=16
  cluster_2=235
  cluster_3=27
Latent break: projectcounty_rcs_te (adj p=0.02848, sig=True)

Selected columns:
['projectcounty_rcs_te', 'naicscode_rcs_te', 'franchisename_rcs_te', 'grossapproval', 'jobssupported', 'borrower_longitude', 'sbadistrictoffice_rcs_te', 'congressionaldistrict_rcs_te', 'businessage_rcs_te', 'borrower_latitude', 'naicscode', 'congressionaldistrict', 'processingmethod_rcs_te', 'structure_id']

Explicit column structures:
Cluster 1: members=['franchisename_rcs_te', 'franchi

In [4]:
from IPython.display import Markdown, display
from structure_id.reporting import export_cluster_profile_report

report_dir = project_root / "reports"
exports = export_cluster_profile_report(
    result=result,
    output_dir=report_dir,
    filename_prefix="training_cluster_profile",
)

print(f"Saved markdown report: {exports['markdown']}")
print(f"Saved HTML report: {exports['html']}")
display(Markdown(exports["markdown"].read_text(encoding="utf-8")))


Saved markdown report: /home/rajiv/programming/structure_identification_for_CS_data/reports/training_cluster_profile.md
Saved HTML report: /home/rajiv/programming/structure_identification_for_CS_data/reports/training_cluster_profile.html


# Training Cluster Profile

## Summary
- Noise rows: 57
- Valid clusters: 4

## Cluster-level interpretation
### Cluster 0
- Size: 35962
- Leading features: borrower_latitude (+0.05sd), borrower_longitude (-0.04sd), sbadistrictoffice_rcs_te (-0.03sd)
- Interpretation: this cluster is generally higher than overall on its strongest discriminating features.

### Cluster 1
- Size: 16
- Leading features: borrower_latitude (-5.68sd), borrower_longitude (+4.83sd), sbadistrictoffice_rcs_te (+3.92sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 2
- Size: 235
- Leading features: borrower_latitude (-5.68sd), borrower_longitude (+4.83sd), sbadistrictoffice_rcs_te (+3.92sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.

### Cluster 3
- Size: 27
- Leading features: borrower_latitude (-5.68sd), projectcounty_rcs_te (+5.46sd), borrower_longitude (+4.83sd)
- Interpretation: this cluster is generally lower than overall on its strongest discriminating features.
